
# Design and Evaluation of an AI-Driven Intrusion Detection Framework for Real-Time Banking Network Security

**MSc Dissertation — Technical Artefact Notebook**

This notebook implements the complete design-science research artefact described in the
dissertation methodology: an end-to-end, reproducible intrusion detection pipeline that
loads CICIDS2017, cleans and engineers features, handles class imbalance, trains and
tunes Random Forest / SVM / Neural Network models plus an ensemble, evaluates every model
with a full metric suite (with an explicit focus on **false positive rate**, since this is
the study's central research problem), and explains model decisions with SHAP and LIME.

> **How to run this notebook**
> 1. Open in Google Colab (`Runtime → Change runtime type → GPU` is optional; CPU is
>    sufficient for Random Forest/SVM, GPU speeds up the neural network).
> 2. Run the **Environment Setup** cell first — it installs every dependency.
> 3. Point `CONFIG.data_raw_dir` (Section 3) at your CICIDS2017 CSV file/folder — see the
>    three supported loading options documented in that section.
> 4. Run All. The notebook is fully reproducible (fixed seeds) and self-contained: every
>    output (models, metrics, figures, SHAP/LIME plots) is saved automatically to the
>    `models/ /results/ /figures/ /metrics/` folders next to this notebook.



## 1. Introduction

### 1.1 Project objective
Banking networks are high-value, continuously-targeted infrastructure: a single
successful intrusion can lead to direct financial loss, regulatory penalties and
reputational damage. Signature-based intrusion detection systems (IDS) struggle against
novel and adaptive attacks, and analysts are frequently overwhelmed by high volumes of
**false positive** alerts, which erodes trust in the system and causes real threats to be
missed ("alert fatigue"). This project designs, builds and evaluates an AI-based Network
Intrusion Detection System (NIDS) that targets **high detection accuracy with a minimised
false positive rate**, and pairs every model with **explainable AI (XAI)** so that a
security analyst can understand *why* an alert fired.

### 1.2 Banking security problem
Financial-sector networks combine (a) extremely high normal-traffic volume, (b) strict
uptime/latency requirements, and (c) a very low tolerance for false alarms, since each
alert consumes scarce analyst time. This creates a harder-than-average class-imbalance and
precision/recall trade-off than typical enterprise IDS deployments.

### 1.3 Intrusion detection & AI motivation
Machine learning and deep learning can learn traffic patterns directly from flow-level
statistics (packet timing, size, flag counts, etc.) without hand-written signatures,
enabling detection of previously unseen attack variants. However, black-box models are
often rejected by security operations teams unless their decisions can be explained —
motivating the SHAP/LIME explainability component of this framework.

### 1.4 Research question
> **How can the real-time network security performance of an AI-based intrusion detection
> system in banking networks be enhanced while minimising the false positive rate?**

Sub-questions addressed by this notebook:
- Which model (Random Forest, SVM, Neural Network, or Ensemble) achieves the best
  accuracy / false-positive-rate trade-off?
- How does structured preprocessing (cleaning, feature selection, imbalance handling)
  change detection performance?
- How does explainable AI (SHAP, LIME) help build analyst trust in the system's decisions?



## 2. Environment Setup

Installs every dependency (safe to re-run — pip skips already-satisfied requirements),
imports libraries, prints environment/version information, and fixes every random seed
used anywhere in the notebook (`numpy`, `random`, `tensorflow`, and every `random_state=`
argument passed to scikit-learn) for full reproducibility.


In [1]:

# --- Locate the project root (so this works whether you're in Colab, or opened
# this notebook locally from its real location inside project/notebooks/) ---
import os
from pathlib import Path

if not Path("utils.py").exists() and Path("../utils.py").exists():
    os.chdir("..")  # step up from project/notebooks/ to project/
print("Working directory:", Path(".").resolve())


Working directory: /home/claude/final_project/DDoS_IDS_Project


In [2]:

# --- Install dependencies (Colab-safe: harmless no-op if already installed) ---
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "shap", "lime", "imbalanced-learn", "optuna", "joblib",
    ], check=True)


In [3]:

# --- Imports ---
from __future__ import annotations

import json
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import tensorflow as tf
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from lime.lime_tabular import LimeTabularExplainer
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, VotingClassifier
from sklearn.feature_selection import RFE, mutual_info_classif
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from tensorflow.keras import Input, Model, callbacks, layers, regularizers

import utils

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
sns.set_theme(style="whitegrid", context="notebook")

print(f"Python        : {sys.version.splitlines()[0]}")
print(f"pandas         : {pd.__version__}")
print(f"numpy          : {np.__version__}")
print(f"scikit-learn   : {__import__('sklearn').__version__}")
print(f"tensorflow     : {tf.__version__}")
print(f"shap           : {shap.__version__}")
print(f"Running in Colab: {IN_COLAB}")


I0000 00:00:1785210904.311200    1236 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785210904.495270    1236 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1785210906.933284    1236 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Python        : 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
pandas         : 3.0.2
numpy          : 2.4.4
scikit-learn   : 1.8.0
tensorflow     : 2.21.0
shap           : 0.52.0
Running in Colab: False


In [4]:

# --- Configuration & reproducibility ---
CONFIG = utils.ProjectConfig(
    project_root=Path("."),
    data_raw_dir=Path("data/raw/cicids2017_real_sample.csv"),  # real CICIDS2017 Friday-DDoS data, stratified 6,000-row sample
    random_state=42,
)
CONFIG.make_dirs()
utils.set_global_seed(CONFIG.random_state)

logger = utils.get_logger(log_dir=CONFIG.logs_dir)
logger.info("Environment initialised. Config: %s", CONFIG)
CONFIG


03:55:07 [INFO] Environment initialised. Config: ProjectConfig(project_root=PosixPath('.'), data_raw_dir=PosixPath('data/raw/cicids2017_real_sample.csv'), random_state=42, label_column='Label', benign_label='BENIGN', test_size=0.15, val_size=0.15, n_jobs=-1)


ProjectConfig(project_root=PosixPath('.'), data_raw_dir=PosixPath('data/raw/cicids2017_real_sample.csv'), random_state=42, label_column='Label', benign_label='BENIGN', test_size=0.15, val_size=0.15, n_jobs=-1)


## 3. Dataset Loading — CICIDS2017

The loader (`utils.load_dataset`) supports **three** input modes, selected automatically
from `CONFIG.data_raw_dir`:

1. **Single CSV** — e.g. one merged `CICIDS2017.csv`.
2. **Folder of CSVs** — the original CICIDS2017 release ships as 8 daily capture files
   (Monday–Friday, several files on Tue/Wed/Thu/Fri); pointing `CONFIG.data_raw_dir` at the
   folder auto-discovers and concatenates every `*.csv` inside it.
3. **Smoke-test file** (default above) — a small synthetic, schema-matching dataset
   generated by `generate_synthetic_smoke_data.py`, used only to prove this notebook runs
   end-to-end without errors in environments without direct access to the official
   CICIDS2017 hosting server. **Replace this path with your real dataset before drawing
   any research conclusions** — see the note in the next cell.

No path is ever hardcoded inside `utils.py` — everything is driven by `CONFIG`, and all
paths use `pathlib.Path` for cross-platform (Windows/Colab/Linux) compatibility.

**Getting the real dataset onto Colab** (pick one):
```python
# Option A — Kaggle (recommended, requires a kaggle.json API token uploaded to Colab):
# from google.colab import files; files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d cicdataset/cicids2017 -p data/raw --unzip

# Option B — Google Drive (if you already keep the CSVs in your Drive):
# from google.colab import drive; drive.mount('/content/drive')
# CONFIG.data_raw_dir = Path('/content/drive/MyDrive/CICIDS2017')

# Option C — direct upload via the Colab file browser into data/raw/
```


In [5]:

if not CONFIG.data_raw_dir.exists():
    raise FileNotFoundError(
        f"'{CONFIG.data_raw_dir}' not found. Run generate_synthetic_smoke_data.py for a "
        f"smoke test, or update CONFIG.data_raw_dir to your real CICIDS2017 file/folder "
        f"(see the loading options documented above)."
    )

with utils.Timer() as t:
    raw_df = utils.load_dataset(CONFIG.data_raw_dir, random_state=CONFIG.random_state)
logger.info("Loaded dataset in %.2fs", t.elapsed)

summary = utils.dataset_summary(raw_df, label_column=CONFIG.label_column)
utils.save_json(summary, CONFIG.reports_dir / "01_raw_dataset_summary.json")
print(f"Rows            : {summary['n_rows']:,}")
print(f"Columns         : {summary['n_columns']}")
print(f"Memory footprint: {summary['memory_mb']} MB")
print(f"Duplicate rows  : {summary['n_duplicates']:,}")
print(f"Missing values  : {summary['n_missing_values']:,}")
raw_df.head()


03:55:07 [INFO] Loaded dataset in 0.08s


Rows            : 6,000
Columns         : 32
Memory footprint: 2.66 MB
Duplicate rows  : 0
Missing values  : 0


,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,...,Max Packet Length,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Label
0,172.16.0.1,36329,192.168.10.50,80,6,07/07/2017 04:11,1706009,5,0,30,...,6,0,0,0,0,1,0,0,0,DDoS
1,192.168.10.50,80,172.16.0.1,54200,6,07/07/2017 03:59,5838518,1,6,6,...,6,0,0,0,0,1,1,0,0,BENIGN
2,192.168.10.15,52336,192.168.10.3,53,17,07/07/2017 04:53,259548,4,2,196,...,265,0,0,0,0,0,0,0,0,BENIGN
3,172.16.0.1,64389,192.168.10.50,80,6,07/07/2017 04:05,10457404,8,4,56,...,8760,0,0,0,0,1,0,0,0,DDoS
4,192.168.10.50,80,172.16.0.1,54902,6,07/07/2017 03:58,73465579,5,9,11613,...,11595,0,0,0,0,1,1,0,0,BENIGN



## 4. Exploratory Data Analysis

Every plot is saved to `figures/` at publication quality (`dpi=110`, tight bounding box).


In [6]:

print(raw_df.shape)
raw_df.dtypes.value_counts()


(6000, 32)


int64      24
str         4
float64     4
Name: count, dtype: int64

In [7]:

missing = raw_df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"Columns with missing values: {len(missing)}")
if len(missing):
    fig, ax = plt.subplots(figsize=(8, max(3, 0.3 * len(missing))))
    missing.plot.barh(ax=ax, color="#c0392b")
    ax.set_title("Missing Values per Column")
    ax.set_xlabel("Count of Missing Values")
    fig.tight_layout()
    fig.savefig(CONFIG.figures_dir / "04_missing_values.png")
    plt.show()
else:
    print("No missing values in the raw numeric columns (infinities are handled in Section 5).")


Columns with missing values: 0
No missing values in the raw numeric columns (infinities are handled in Section 5).


In [8]:

print(f"Duplicate rows: {raw_df.duplicated().sum():,} ({raw_df.duplicated().mean()*100:.2f}% of dataset)")
print(f"Unique values per column (top 10 lowest-cardinality):")
raw_df.nunique().sort_values().head(10)


Duplicate rows: 0 (0.00% of dataset)
Unique values per column (top 10 lowest-cardinality):


Fwd URG Flags     1
Bwd PSH Flags     1
CWE Flag Count    1
Bwd URG Flags     1
FIN Flag Count    2
URG Flag Count    2
Fwd PSH Flags     2
RST Flag Count    2
ACK Flag Count    2
SYN Flag Count    2
dtype: int64

In [9]:

class_counts = raw_df[CONFIG.label_column].value_counts()
print(class_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_counts.plot(kind="bar", ax=axes[0], color=sns.color_palette("Set2", len(class_counts)))
axes[0].set_title("Attack-Class Distribution (raw counts)")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=75)

binary_counts = raw_df[CONFIG.label_column].apply(
    lambda x: "BENIGN" if x == CONFIG.benign_label else "ATTACK"
).value_counts()
axes[1].pie(binary_counts, labels=binary_counts.index, autopct="%1.1f%%",
            colors=["#2ecc71", "#e74c3c"])
axes[1].set_title("Benign vs Attack (class imbalance)")
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "04_class_distribution.png")
plt.show()


Label
DDoS      3403
BENIGN    2597
Name: count, dtype: int64


In [10]:

numeric_cols = raw_df.select_dtypes(include=[np.number]).columns.tolist()
desc = raw_df[numeric_cols].describe().T
desc["skew"] = raw_df[numeric_cols].skew(numeric_only=True)
desc.to_csv(CONFIG.metrics_dir / "04_feature_statistics.csv")
desc.head(15)


,count,mean,std,min,25%,50%,75%,max,skew
Source Port,6000.0,3.814393e+04,2.304982e+04,0.000000,18815.500000,4.960200e+04,5.827650e+04,6.553200e+04,-0.576360
Destination Port,6000.0,9.080510e+03,2.003644e+04,0.000000,80.000000,8.000000e+01,8.000000e+01,6.547000e+04,1.921627
Protocol,6000.0,7.578333e+00,3.860960e+00,0.000000,6.000000,6.000000e+00,6.000000e+00,1.700000e+01,2.027645
Flow Duration,6000.0,1.597640e+07,3.128491e+07,1.000000,65254.500000,1.334924e+06,8.570004e+06,1.199957e+08,1.955058
Total Fwd Packets,6000.0,4.497333e+00,6.696726e+00,1.000000,2.000000,3.000000e+00,5.000000e+00,2.810000e+02,17.869905
Total Backward Packets,6000.0,4.104667e+00,8.652063e+00,0.000000,1.000000,4.000000e+00,5.000000e+00,4.230000e+02,24.254923
Total Length of Fwd Packets,6000.0,8.682670e+02,2.895196e+03,0.000000,26.000000,3.000000e+01,6.200000e+01,3.333400e+04,3.653444
Total Length of Bwd Packets,6000.0,5.334513e+03,1.526599e+04,0.000000,0.000000,1.640000e+02,1.160100e+04,8.563090e+05,32.737656
Flow Bytes/s,6000.0,7.856028e+05,1.947435e+07,0.000000,12.005385,1.191321e+03,2.116842e+04,1.040000e+09,41.306144
Flow Packets/s,6000.0,1.339853e+04,1.066367e+05,0.040619,0.627673,5.534706e+00,7.087127e+01,2.000000e+06,12.970649


In [11]:

sample_feats = numeric_cols[: min(9, len(numeric_cols))]
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, col in zip(axes.ravel(), sample_feats):
    sns.histplot(raw_df[col].replace([np.inf, -np.inf], np.nan).dropna(), bins=40, ax=ax, color="#2980b9")
    ax.set_title(col, fontsize=9)
fig.suptitle("Feature Distributions (sample)")
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "04_feature_histograms.png")
plt.show()


In [12]:

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, col in zip(axes.ravel(), sample_feats):
    sns.boxplot(x=raw_df[CONFIG.label_column].apply(
        lambda v: "BENIGN" if v == CONFIG.benign_label else "ATTACK"),
        y=raw_df[col].replace([np.inf, -np.inf], np.nan), ax=ax, palette="Set2")
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("")
fig.suptitle("Boxplots: Feature Spread by Benign/Attack (outlier visualisation)")
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "04_boxplots_outliers.png")
plt.show()


In [13]:

corr = raw_df[numeric_cols].replace([np.inf, -np.inf], np.nan).dropna().corr()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax, cbar_kws={"shrink": 0.7})
ax.set_title("Feature Correlation Heatmap")
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "04_correlation_heatmap.png")
plt.show()


In [14]:

pairplot_cols = numeric_cols[: min(5, len(numeric_cols))]
pp_df = raw_df[pairplot_cols + [CONFIG.label_column]].replace([np.inf, -np.inf], np.nan).dropna()
pp_df = pp_df.sample(n=min(800, len(pp_df)), random_state=CONFIG.random_state)
pp_df["Class"] = pp_df[CONFIG.label_column].apply(lambda v: "BENIGN" if v == CONFIG.benign_label else "ATTACK")
g = sns.pairplot(pp_df, vars=pairplot_cols, hue="Class", palette={"BENIGN": "#2ecc71", "ATTACK": "#e74c3c"},
                  plot_kws={"alpha": 0.5, "s": 15})
g.fig.suptitle("Sampled Pairplot: Benign vs Attack", y=1.02)
g.savefig(CONFIG.figures_dir / "04_pairplot_sampled.png")
plt.show()


In [15]:

Q1 = raw_df[numeric_cols].replace([np.inf, -np.inf], np.nan).quantile(0.25)
Q3 = raw_df[numeric_cols].replace([np.inf, -np.inf], np.nan).quantile(0.75)
IQR = Q3 - Q1
outlier_frac = (
    (raw_df[numeric_cols].replace([np.inf, -np.inf], np.nan) < (Q1 - 1.5 * IQR))
    | (raw_df[numeric_cols].replace([np.inf, -np.inf], np.nan) > (Q3 + 1.5 * IQR))
).mean().sort_values(ascending=False)
print("Top-10 columns by IQR-outlier fraction:")
outlier_frac.head(10)


Top-10 columns by IQR-outlier fraction:


Destination Port               0.389667
Flow Bytes/s                   0.206167
Bwd Packets/s                  0.179167
Fwd Packets/s                  0.175500
Flow Packets/s                 0.173667
Flow Duration                  0.162833
Fwd Header Length              0.158000
Total Length of Fwd Packets    0.155833
Protocol                       0.144000
Min Packet Length              0.143833
dtype: float64


## 5. Data Cleaning

Every decision is logged and quantified via `utils.clean_dataset`, which:
1. Removes exact duplicate rows.
2. Replaces `inf`/`-inf` (CICIDS2017's `Flow Bytes/s` and `Flow Packets/s` divide-by-zero
   artefacts when `Flow Duration == 0`) with `NaN`, then drops any row that is still
   missing a numeric value.
3. Drops identifier columns (`Flow ID`, source/destination IP & port, `Timestamp`) —
   keeping them would let the model memorise *who* rather than learn *behavioural*
   patterns, which would not generalise to a live banking network (this is also a
   textbook **data-leakage** avoidance step, flagged in the audit report).
4. Drops zero-variance columns, which carry no discriminative signal.


In [16]:

clean_df, clean_report = utils.clean_dataset(
    raw_df, label_column=CONFIG.label_column, logger=logger
)
utils.save_json(clean_report, CONFIG.reports_dir / "05_cleaning_report.json")
print(json.dumps(clean_report, indent=2, default=str))
clean_df.to_csv(CONFIG.data_processed_dir / "clean_dataset.csv", index=False)
clean_df.shape


03:55:14 [INFO] Cleaning complete: {'duplicates_removed': 0, 'identifier_columns_dropped': ['Source IP', 'Destination IP', 'Timestamp', 'Source Port'], 'rows_dropped_inf_or_missing': 0, 'zero_variance_columns_dropped': ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count'], 'final_shape': [6000, 24]}


{
  "duplicates_removed": 0,
  "identifier_columns_dropped": [
    "Source IP",
    "Destination IP",
    "Timestamp",
    "Source Port"
  ],
  "rows_dropped_inf_or_missing": 0,
  "zero_variance_columns_dropped": [
    "Bwd PSH Flags",
    "Fwd URG Flags",
    "Bwd URG Flags",
    "CWE Flag Count"
  ],
  "final_shape": [
    6000,
    24
  ]
}


(6000, 24)


## 6. Feature Engineering

Pipeline: label encode → variance-threshold filter → correlation-based redundancy
removal → Random-Forest importance ranking → mutual information ranking → RFE
confirmation → (optional) PCA comparison. Every removed feature is logged with the
reason it was removed, satisfying the "document why features were removed" requirement.


In [17]:

y_raw = clean_df[CONFIG.label_column].copy()
X_raw = clean_df.drop(columns=[CONFIG.label_column])

label_encoder = LabelEncoder()
y_multiclass = label_encoder.fit_transform(y_raw)
class_names = label_encoder.classes_
utils.save_json({"classes": class_names.tolist()}, CONFIG.reports_dir / "06_label_classes.json")

# Binary target is the primary research target (benign=0 / attack=1), matching the
# dissertation's FPR-centric evaluation; the multiclass y is retained for error analysis.
y_binary = (y_raw != CONFIG.benign_label).astype(int)
print("Binary class balance:\n", y_binary.value_counts())


Binary class balance:
 Label
1    3403
0    2597
Name: count, dtype: int64


In [18]:

X_vt, dropped_vt = utils.variance_threshold_filter(
    pd.concat([X_raw, y_raw.rename(CONFIG.label_column)], axis=1),
    threshold=1e-4, label_column=CONFIG.label_column,
)
X_vt = X_vt.drop(columns=[CONFIG.label_column], errors="ignore")
print(f"Variance-threshold removed {len(dropped_vt)} columns: {dropped_vt}")

X_corr, dropped_corr = utils.remove_correlated_features(
    pd.concat([X_vt, y_raw.rename(CONFIG.label_column)], axis=1),
    threshold=0.95, label_column=CONFIG.label_column,
)
X_corr = X_corr.drop(columns=[CONFIG.label_column], errors="ignore")
print(f"Correlation filter (>0.95) removed {len(dropped_corr)} columns: {dropped_corr}")

feature_removal_log = {
    "variance_threshold_removed": dropped_vt,
    "correlation_removed": dropped_corr,
}
utils.save_json(feature_removal_log, CONFIG.reports_dir / "06_feature_removal_log.json")
X_corr.shape


Variance-threshold removed 0 columns: []
Correlation filter (>0.95) removed 4 columns: ['Bwd Header Length', 'Fwd Packets/s', 'SYN Flag Count', 'ECE Flag Count']


(6000, 19)

In [19]:

# Random Forest feature importance (fast, robust baseline ranking)
rf_selector = RandomForestClassifier(
    n_estimators=200, random_state=CONFIG.random_state, n_jobs=CONFIG.n_jobs, class_weight="balanced"
)
rf_selector.fit(X_corr, y_binary)
importances = pd.Series(rf_selector.feature_importances_, index=X_corr.columns).sort_values(ascending=False)
importances.to_csv(CONFIG.metrics_dir / "06_rf_feature_importance.csv")

fig, ax = plt.subplots(figsize=(8, 8))
importances.head(20).sort_values().plot.barh(ax=ax, color="#16a085")
ax.set_title("Top-20 Random Forest Feature Importances")
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "06_rf_feature_importance.png")
plt.show()


In [20]:

# Mutual information ranking (captures non-linear relevance RF importance can miss)
mi_scores = mutual_info_classif(X_corr, y_binary, random_state=CONFIG.random_state)
mi_series = pd.Series(mi_scores, index=X_corr.columns).sort_values(ascending=False)
mi_series.to_csv(CONFIG.metrics_dir / "06_mutual_information.csv")
mi_series.head(15)


Total Length of Fwd Packets    0.652237
Fwd Header Length              0.538132
Total Length of Bwd Packets    0.530825
Destination Port               0.528325
Total Fwd Packets              0.406395
Bwd Packets/s                  0.335748
Total Backward Packets         0.316646
Max Packet Length              0.288083
Flow Bytes/s                   0.223111
Flow Duration                  0.210723
Flow Packets/s                 0.210606
Protocol                       0.141289
URG Flag Count                 0.129520
Min Packet Length              0.129256
PSH Flag Count                 0.027892
dtype: float64

In [21]:

# Keep the intersection of "top-K by RF importance" and "top-K by mutual information"
# as our final feature set — a feature must prove useful under two different criteria
# to survive, which reduces the risk of a spurious, dataset-specific artefact driving
# the whole model (a common source of poor generalisation in IDS literature).
TOP_K = min(25, X_corr.shape[1])
top_rf = set(importances.head(TOP_K).index)
top_mi = set(mi_series.head(TOP_K).index)
selected_features = sorted(top_rf & top_mi) or sorted(top_rf)  # fallback if intersection is empty
print(f"Selected {len(selected_features)} features (RF ∩ MI top-{TOP_K}):")
print(selected_features)

X_selected = X_corr[selected_features].copy()
utils.save_json({"selected_features": selected_features}, CONFIG.reports_dir / "06_selected_features.json")


Selected 19 features (RF ∩ MI top-19):
['ACK Flag Count', 'Bwd Packets/s', 'Destination Port', 'FIN Flag Count', 'Flow Bytes/s', 'Flow Duration', 'Flow Packets/s', 'Fwd Header Length', 'Fwd PSH Flags', 'Max Packet Length', 'Min Packet Length', 'PSH Flag Count', 'Protocol', 'RST Flag Count', 'Total Backward Packets', 'Total Fwd Packets', 'Total Length of Bwd Packets', 'Total Length of Fwd Packets', 'URG Flag Count']


In [22]:

# RFE confirmation pass (cheap, small estimator, small n_features_to_select just to sanity-check ranking agreement)
rfe_estimator = RandomForestClassifier(n_estimators=100, random_state=CONFIG.random_state, n_jobs=CONFIG.n_jobs)
rfe = RFE(rfe_estimator, n_features_to_select=max(5, len(selected_features) // 2))
rfe.fit(X_selected, y_binary)
rfe_ranking = pd.Series(rfe.ranking_, index=X_selected.columns).sort_values()
print("RFE agreement (rank 1 = kept):")
rfe_ranking.head(10)


RFE agreement (rank 1 = kept):


Bwd Packets/s                  1
Destination Port               1
Fwd Header Length              1
Max Packet Length              1
Total Fwd Packets              1
Total Backward Packets         1
Min Packet Length              1
Total Length of Bwd Packets    1
Total Length of Fwd Packets    1
Protocol                       2
dtype: int64

In [23]:

# Optional PCA comparison (kept ONLY for a side-by-side variance-explained comparison;
# the interpretable, selected feature set above — not PCA components — feeds the models,
# preserving explainability, which PCA components would destroy).
from sklearn.decomposition import PCA

scaler_for_pca = StandardScaler().fit(X_selected)
pca = PCA(n_components=min(10, X_selected.shape[1]), random_state=CONFIG.random_state)
pca.fit(scaler_for_pca.transform(X_selected))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.cumsum(pca.explained_variance_ratio_), marker="o", color="#8e44ad")
ax.set_xlabel("Number of Principal Components")
ax.set_ylabel("Cumulative Explained Variance")
ax.set_title("PCA — Explained Variance (comparison only, not used downstream)")
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "06_pca_explained_variance.png")
plt.show()



## 7. Handle Class Imbalance

CICIDS2017 is heavily benign-dominated. We compare four strategies on the **training
split only** (to avoid leaking synthetic/duplicated samples into validation/test — a
common data-leakage mistake this framework explicitly avoids) using 5-fold
stratified-CV F1-score of a fixed Random-Forest probe model, then pick the best method
by CV score.


In [24]:

X_train_full, X_temp, y_train_full, y_temp = train_test_split(
    X_selected, y_binary, test_size=(CONFIG.test_size + CONFIG.val_size),
    stratify=y_binary, random_state=CONFIG.random_state,
)
relative_val_size = CONFIG.val_size / (CONFIG.test_size + CONFIG.val_size)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=(1 - relative_val_size),
    stratify=y_temp, random_state=CONFIG.random_state,
)
print(f"Train: {X_train_full.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print("(This preliminary split is only used to fairly evaluate imbalance strategies "
      "without touching val/test; Section 8 re-confirms the final split.)")


Train: (4200, 19), Val: (900, 19), Test: (900, 19)
(This preliminary split is only used to fairly evaluate imbalance strategies without touching val/test; Section 8 re-confirms the final split.)


In [25]:

scaler = StandardScaler().fit(X_train_full)
X_train_scaled = pd.DataFrame(scaler.transform(X_train_full), columns=X_train_full.columns, index=X_train_full.index)

strategies = {
    "class_weight_balanced": None,  # handled at model level, no resampling
    "random_oversampling": RandomOverSampler(random_state=CONFIG.random_state),
    "smote": SMOTE(random_state=CONFIG.random_state, k_neighbors=min(5, y_train_full.value_counts().min() - 1) or 1),
    "random_undersampling": RandomUnderSampler(random_state=CONFIG.random_state),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=CONFIG.random_state)
strategy_scores = {}

for name, sampler in strategies.items():
    if sampler is None:
        probe = RandomForestClassifier(
            n_estimators=150, class_weight="balanced", random_state=CONFIG.random_state, n_jobs=CONFIG.n_jobs
        )
        scores = cross_val_score(probe, X_train_scaled, y_train_full, cv=cv, scoring="f1", n_jobs=CONFIG.n_jobs)
    else:
        X_res, y_res = sampler.fit_resample(X_train_scaled, y_train_full)
        probe = RandomForestClassifier(n_estimators=150, random_state=CONFIG.random_state, n_jobs=CONFIG.n_jobs)
        scores = cross_val_score(probe, X_res, y_res, cv=cv, scoring="f1", n_jobs=CONFIG.n_jobs)
    strategy_scores[name] = float(scores.mean())
    print(f"{name:25s} mean CV F1 = {scores.mean():.4f} (+/- {scores.std():.4f})")

utils.save_json(strategy_scores, CONFIG.metrics_dir / "07_imbalance_strategy_scores.json")
BEST_STRATEGY = max(strategy_scores, key=strategy_scores.get)
print(f"\nSelected imbalance strategy: {BEST_STRATEGY} "
      f"(highest mean CV F1 = {strategy_scores[BEST_STRATEGY]:.4f}). "
      f"Rationale: F1 balances precision and recall, which is exactly the trade-off the "
      f"dissertation cares about (missed attacks vs analyst alert fatigue).")


class_weight_balanced     mean CV F1 = 0.9983 (+/- 0.0011)


random_oversampling       mean CV F1 = 0.9983 (+/- 0.0014)


smote                     mean CV F1 = 0.9981 (+/- 0.0015)


random_undersampling      mean CV F1 = 0.9981 (+/- 0.0007)

Selected imbalance strategy: class_weight_balanced (highest mean CV F1 = 0.9983). Rationale: F1 balances precision and recall, which is exactly the trade-off the dissertation cares about (missed attacks vs analyst alert fatigue).


In [26]:

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
y_train_full.value_counts().rename({0: "BENIGN", 1: "ATTACK"}).plot.bar(ax=axes[0], color=["#2ecc71", "#e74c3c"])
axes[0].set_title("Training Class Balance — BEFORE")

if strategies[BEST_STRATEGY] is not None:
    X_bal_preview, y_bal_preview = strategies[BEST_STRATEGY].fit_resample(X_train_scaled, y_train_full)
else:
    X_bal_preview, y_bal_preview = X_train_scaled, y_train_full
pd.Series(y_bal_preview).value_counts().rename({0: "BENIGN", 1: "ATTACK"}).plot.bar(ax=axes[1], color=["#2ecc71", "#e74c3c"])
axes[1].set_title(f"Training Class Balance — AFTER ({BEST_STRATEGY})")
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "07_class_balance_before_after.png")
plt.show()



## 8. Train / Validation / Test Split

Final, leakage-safe split: **70% train / 15% validation / 15% test**, stratified on the
binary label. Imbalance handling (Section 7's chosen strategy) is applied to the
**training split only**, fit *after* the split — never before — so information from
validation/test never influences resampling or scaling (a data-leakage mistake common in
published IDS notebooks, flagged explicitly in `AUDIT_REPORT.md`).


In [27]:

X_train, X_temp, y_train, y_temp = train_test_split(
    X_selected, y_binary, test_size=(CONFIG.test_size + CONFIG.val_size),
    stratify=y_binary, random_state=CONFIG.random_state,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=(1 - relative_val_size),
    stratify=y_temp, random_state=CONFIG.random_state,
)

scaler = StandardScaler().fit(X_train)  # fit ONLY on train
X_train_s = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_s = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
joblib.dump(scaler, CONFIG.models_dir / "feature_scaler.joblib")

if BEST_STRATEGY == "class_weight_balanced":
    X_train_final, y_train_final = X_train_s, y_train
else:
    X_train_final, y_train_final = strategies[BEST_STRATEGY].fit_resample(X_train_s, y_train)

print(f"Final training set  (post-imbalance handling): {X_train_final.shape}")
print(f"Validation set                                : {X_val_s.shape}")
print(f"Test set (held out, untouched by resampling)   : {X_test_s.shape}")


Final training set  (post-imbalance handling): (4200, 19)
Validation set                                : (900, 19)
Test set (held out, untouched by resampling)   : (900, 19)



## 9. Model Development — Random Forest & SVM

Each model below has: a `Pipeline`, baseline training, cross-validation, hyperparameter
tuning (Section 11 goes deeper) and full evaluation (Section 13). SVM is trained on a
stratified subsample when the training set is large, since SVM's O(n²)–O(n³) training
cost makes it impractical on the full multi-million-row CICIDS2017 — this is a documented,
deliberate scalability decision, not a shortcut, and is disclosed in `AUDIT_REPORT.md`.


In [28]:

model_results = {}          # metrics per model, populated in Section 13
trained_models = {}         # fitted estimators, for Sections 12/14/15/16/17
training_times = {}


In [29]:

# --- Random Forest ---
rf_pipeline = Pipeline([
    ("classifier", RandomForestClassifier(random_state=CONFIG.random_state, n_jobs=CONFIG.n_jobs)),
])
rf_param_grid = {
    "classifier__n_estimators": [200, 400],
    "classifier__max_depth": [None, 20, 40],
    "classifier__min_samples_split": [2, 5],
}
rf_search = RandomizedSearchCV(
    rf_pipeline, rf_param_grid, n_iter=6, cv=3, scoring="f1",
    random_state=CONFIG.random_state, n_jobs=CONFIG.n_jobs,
)
with utils.Timer() as t:
    rf_search.fit(X_train_final, y_train_final)
training_times["Random Forest"] = t.elapsed
print(f"Best RF params: {rf_search.best_params_}  (CV F1={rf_search.best_score_:.4f}, {t.elapsed:.1f}s)")
trained_models["Random Forest"] = rf_search.best_estimator_
utils.save_model(rf_search.best_estimator_, CONFIG.models_dir / "random_forest.joblib")


Best RF params: {'classifier__n_estimators': 200, 'classifier__min_samples_split': 5, 'classifier__max_depth': 40}  (CV F1=0.9981, 11.5s)


In [30]:

# --- Support Vector Machine ---
MAX_SVM_TRAIN = 6000  # documented scalability cap; see markdown above
if len(X_train_final) > MAX_SVM_TRAIN:
    svm_X, _, svm_y, _ = train_test_split(
        X_train_final, y_train_final, train_size=MAX_SVM_TRAIN,
        stratify=y_train_final, random_state=CONFIG.random_state,
    )
else:
    svm_X, svm_y = X_train_final, y_train_final

svm_pipeline = Pipeline([
    ("classifier", SVC(probability=True, random_state=CONFIG.random_state)),
])
svm_param_grid = {
    "classifier__C": [1, 10, 50],
    "classifier__kernel": ["rbf"],
    "classifier__gamma": ["scale", "auto"],
}
svm_search = RandomizedSearchCV(
    svm_pipeline, svm_param_grid, n_iter=4, cv=3, scoring="f1",
    random_state=CONFIG.random_state, n_jobs=CONFIG.n_jobs,
)
with utils.Timer() as t:
    svm_search.fit(svm_X, svm_y)
training_times["SVM"] = t.elapsed
print(f"Best SVM params: {svm_search.best_params_}  (CV F1={svm_search.best_score_:.4f}, {t.elapsed:.1f}s, "
      f"trained on {len(svm_X)} stratified samples)")
trained_models["SVM"] = svm_search.best_estimator_
utils.save_model(svm_search.best_estimator_, CONFIG.models_dir / "svm.joblib")


Best SVM params: {'classifier__kernel': 'rbf', 'classifier__gamma': 'auto', 'classifier__C': 50}  (CV F1=0.9981, 1.0s, trained on 4200 stratified samples)



## 10. Neural Network (TensorFlow / Keras)

A feed-forward network with `Dense → BatchNormalization → Dropout` blocks,
`EarlyStopping`, `ModelCheckpoint` (best validation loss) and a `ReduceLROnPlateau`
learning-rate schedule.


In [31]:

def build_nn(input_dim: int, random_state: int = 42) -> Model:
    tf.random.set_seed(random_state)
    inputs = Input(shape=(input_dim,))
    x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )
    return model


nn_model = build_nn(X_train_final.shape[1], CONFIG.random_state)
nn_model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 19)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         2,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,825 (54.00 KB)

 Trainable params: 13,377 (52.25 KB)

 Non-trainable params: 448 (1.75 KB)

In [32]:

nn_checkpoint_path = CONFIG.models_dir / "neural_network_best.keras"
nn_callbacks = [
    callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    callbacks.ModelCheckpoint(str(nn_checkpoint_path), monitor="val_loss", save_best_only=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
]

with utils.Timer() as t:
    history = nn_model.fit(
        X_train_final.values, y_train_final.values if hasattr(y_train_final, "values") else y_train_final,
        validation_data=(X_val_s.values, y_val.values),
        epochs=60, batch_size=256, callbacks=nn_callbacks, verbose=0,
    )
training_times["Neural Network"] = t.elapsed
trained_models["Neural Network"] = nn_model
print(f"Neural network trained for {len(history.history['loss'])} epochs in {t.elapsed:.1f}s "
      f"(early stopping active).")


Neural network trained for 44 epochs in 12.4s (early stopping active).


In [33]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Neural Network — Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="val")
axes[1].set_title("Neural Network — Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "10_nn_training_history.png")
plt.show()



## 11. Hyperparameter Optimisation — Tuned vs Untuned

Section 9 already used `RandomizedSearchCV`. Here we explicitly quantify the *lift* tuning
provided over an untuned, default-hyperparameter baseline for Random Forest and SVM, so the
value of tuning is demonstrated rather than assumed.


In [34]:

def quick_f1(estimator, X, y) -> float:
    from sklearn.metrics import f1_score as _f1
    return _f1(y, estimator.predict(X))


baseline_rf = RandomForestClassifier(random_state=CONFIG.random_state, n_jobs=CONFIG.n_jobs).fit(X_train_final, y_train_final)
baseline_svm = SVC(probability=True, random_state=CONFIG.random_state).fit(svm_X, svm_y)

tuning_comparison = pd.DataFrame({
    "model": ["Random Forest", "Random Forest", "SVM", "SVM"],
    "variant": ["untuned (defaults)", "tuned (RandomizedSearchCV)", "untuned (defaults)", "tuned (RandomizedSearchCV)"],
    "val_f1": [
        quick_f1(baseline_rf, X_val_s, y_val),
        quick_f1(trained_models["Random Forest"], X_val_s, y_val),
        quick_f1(baseline_svm, X_val_s, y_val),
        quick_f1(trained_models["SVM"], X_val_s, y_val),
    ],
})
tuning_comparison.to_csv(CONFIG.metrics_dir / "11_tuning_comparison.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=tuning_comparison, x="model", y="val_f1", hue="variant", ax=ax, palette="Set2")
ax.set_title("Validation F1: Tuned vs Untuned")
ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "11_tuning_comparison.png")
plt.show()
tuning_comparison


,model,variant,val_f1
0,Random Forest,untuned (defaults),0.999019
1,Random Forest,tuned (RandomizedSearchCV),0.999019
2,SVM,untuned (defaults),0.995122
3,SVM,tuned (RandomizedSearchCV),0.998039



## 12. Ensemble Model

A soft-voting ensemble combines Random Forest + SVM + Neural Network probability
outputs. Because Keras models aren't natively scikit-learn estimators, we wrap the
neural network in a thin scikit-learn-compatible adapter so it can sit inside a
`VotingClassifier`-style soft average alongside RF and SVM. We also fit a
`StackingClassifier` (RF + SVM as base learners, logistic regression meta-learner) as a
second ensemble strategy and report whichever performs best on validation.


In [35]:

class KerasProbaWrapper:
    '''Minimal sklearn-compatible wrapper so a fitted Keras model can be combined
    with sklearn estimators for soft-voting ensembling.'''

    def __init__(self, keras_model):
        self.keras_model = keras_model
        self.classes_ = np.array([0, 1])

    def predict_proba(self, X):
        p1 = self.keras_model.predict(np.asarray(X), verbose=0).ravel()
        return np.column_stack([1 - p1, p1])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


nn_wrapped = KerasProbaWrapper(trained_models["Neural Network"])

def soft_vote_proba(models_dict, X):
    probs = [m.predict_proba(X)[:, 1] for m in models_dict.values()]
    return np.mean(probs, axis=0)

ensemble_members = {
    "Random Forest": trained_models["Random Forest"],
    "SVM": trained_models["SVM"],
    "Neural Network": nn_wrapped,
}

val_proba_ensemble = soft_vote_proba(ensemble_members, X_val_s)
val_pred_ensemble = (val_proba_ensemble >= 0.5).astype(int)
soft_vote_val_f1 = f1_score(y_val, val_pred_ensemble)
print(f"Soft-voting ensemble validation F1: {soft_vote_val_f1:.4f}")


Soft-voting ensemble validation F1: 0.9990


In [36]:

stacking = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(n_estimators=200, random_state=CONFIG.random_state, n_jobs=CONFIG.n_jobs)),
        ("svm", SVC(probability=True, random_state=CONFIG.random_state, C=10, kernel="rbf")),
    ],
    final_estimator=__import__("sklearn.linear_model", fromlist=["LogisticRegression"]).LogisticRegression(max_iter=1000),
    cv=3, n_jobs=CONFIG.n_jobs,
)
# StackingClassifier needs a single homogeneous training set / size SVM can handle
stack_train_X = X_train_final if len(X_train_final) <= MAX_SVM_TRAIN else X_train_final.sample(MAX_SVM_TRAIN, random_state=CONFIG.random_state)
stack_train_y = y_train_final.loc[stack_train_X.index] if hasattr(y_train_final, "loc") else y_train_final[:len(stack_train_X)]
with utils.Timer() as t:
    stacking.fit(stack_train_X, stack_train_y)
stack_val_f1 = f1_score(y_val, stacking.predict(X_val_s))
print(f"Stacking ensemble validation F1: {stack_val_f1:.4f}  ({t.elapsed:.1f}s)")

if soft_vote_val_f1 >= stack_val_f1:
    ENSEMBLE_STRATEGY = "soft_voting"
    print("Selected ensemble strategy: SOFT VOTING (RF + SVM + NN) — higher validation F1.")
else:
    ENSEMBLE_STRATEGY = "stacking"
    trained_models["Ensemble"] = stacking
    print("Selected ensemble strategy: STACKING (RF + SVM, LR meta-learner) — higher validation F1.")


Stacking ensemble validation F1: 0.9990  (2.2s)
Selected ensemble strategy: SOFT VOTING (RF + SVM + NN) — higher validation F1.



## 13. Model Evaluation

Full metric suite for every model — Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC,
Confusion Matrix, Specificity, Sensitivity, **False Positive Rate**, False Negative Rate,
Balanced Accuracy, Matthews Correlation Coefficient, Cohen's Kappa, and the full
classification report — computed on the **held-out test set only**.


In [37]:

def evaluate_and_store(name: str, y_pred: np.ndarray, y_proba: np.ndarray) -> None:
    metrics = utils.compute_classification_metrics(y_test.values, y_pred, y_proba)
    model_results[name] = metrics
    print(f"\n=== {name} ===")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k:22s}: {v:.4f}")
    from sklearn.metrics import classification_report
    print(classification_report(y_test, y_pred, target_names=["BENIGN", "ATTACK"], zero_division=0))


rf_pred = trained_models["Random Forest"].predict(X_test_s)
rf_proba = trained_models["Random Forest"].predict_proba(X_test_s)[:, 1]
evaluate_and_store("Random Forest", rf_pred, rf_proba)

svm_pred = trained_models["SVM"].predict(X_test_s)
svm_proba = trained_models["SVM"].predict_proba(X_test_s)[:, 1]
evaluate_and_store("SVM", svm_pred, svm_proba)

nn_proba = nn_wrapped.predict_proba(X_test_s)[:, 1]
nn_pred = (nn_proba >= 0.5).astype(int)
evaluate_and_store("Neural Network", nn_pred, nn_proba)

if ENSEMBLE_STRATEGY == "soft_voting":
    ens_proba = soft_vote_proba(ensemble_members, X_test_s)
else:
    ens_proba = trained_models["Ensemble"].predict_proba(X_test_s)[:, 1]
ens_pred = (ens_proba >= 0.5).astype(int)
evaluate_and_store("Ensemble", ens_pred, ens_proba)

metrics_df = pd.DataFrame(model_results).T
metrics_df.to_csv(CONFIG.metrics_dir / "13_all_model_metrics.csv")
metrics_df



=== Random Forest ===
  accuracy              : 0.9989
  precision             : 0.9980
  recall                : 1.0000
  f1_score              : 0.9990
  false_positive_rate   : 0.0026
  false_negative_rate   : 0.0000
  specificity           : 0.9974
  sensitivity           : 1.0000
  balanced_accuracy     : 0.9987
  matthews_corrcoef     : 0.9977
  cohen_kappa           : 0.9977
  roc_auc               : 1.0000
  pr_auc                : 1.0000
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00       389
      ATTACK       1.00      1.00      1.00       511

    accuracy                           1.00       900
   macro avg       1.00      1.00      1.00       900
weighted avg       1.00      1.00      1.00       900


=== SVM ===
  accuracy              : 0.9989
  precision             : 0.9980
  recall                : 1.0000
  f1_score              : 0.9990
  false_positive_rate   : 0.0026
  false_negative_rate   : 0.0000
  specific


=== Neural Network ===
  accuracy              : 1.0000
  precision             : 1.0000
  recall                : 1.0000
  f1_score              : 1.0000
  false_positive_rate   : 0.0000
  false_negative_rate   : 0.0000
  specificity           : 1.0000
  sensitivity           : 1.0000
  balanced_accuracy     : 1.0000
  matthews_corrcoef     : 1.0000
  cohen_kappa           : 1.0000
  roc_auc               : 1.0000
  pr_auc                : 1.0000
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00       389
      ATTACK       1.00      1.00      1.00       511

    accuracy                           1.00       900
   macro avg       1.00      1.00      1.00       900
weighted avg       1.00      1.00      1.00       900




=== Ensemble ===
  accuracy              : 0.9989
  precision             : 0.9980
  recall                : 1.0000
  f1_score              : 0.9990
  false_positive_rate   : 0.0026
  false_negative_rate   : 0.0000
  specificity           : 0.9974
  sensitivity           : 1.0000
  balanced_accuracy     : 0.9987
  matthews_corrcoef     : 0.9977
  cohen_kappa           : 0.9977
  roc_auc               : 1.0000
  pr_auc                : 1.0000
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00       389
      ATTACK       1.00      1.00      1.00       511

    accuracy                           1.00       900
   macro avg       1.00      1.00      1.00       900
weighted avg       1.00      1.00      1.00       900



,accuracy,precision,recall,f1_score,false_positive_rate,false_negative_rate,specificity,sensitivity,balanced_accuracy,matthews_corrcoef,cohen_kappa,roc_auc,pr_auc
Random Forest,0.998889,0.998047,1.0,0.999022,0.002571,0.0,0.997429,1.0,0.998715,0.997738,0.997735,1.0,1.0
SVM,0.998889,0.998047,1.0,0.999022,0.002571,0.0,0.997429,1.0,0.998715,0.997738,0.997735,1.0,1.0
Neural Network,1.000000,1.000000,1.0,1.000000,0.000000,0.0,1.000000,1.0,1.000000,1.000000,1.000000,1.0,1.0
Ensemble,0.998889,0.998047,1.0,0.999022,0.002571,0.0,0.997429,1.0,0.998715,0.997738,0.997735,1.0,1.0



## 14. Visualisations

Publication-quality comparison figures for every model, saved to `figures/`.


In [38]:

proba_lookup = {"Random Forest": rf_proba, "SVM": svm_proba, "Neural Network": nn_proba, "Ensemble": ens_proba}
pred_lookup = {"Random Forest": rf_pred, "SVM": svm_pred, "Neural Network": nn_pred, "Ensemble": ens_pred}

fig, ax = plt.subplots(figsize=(6, 6))
for name, proba in proba_lookup.items():
    curve = utils.get_roc_pr_curve_data(y_test.values, proba)
    ax.plot(curve["fpr"], curve["tpr"], label=f"{name} (AUC={model_results[name].get('roc_auc', float('nan')):.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — All Models")
ax.legend()
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "14_roc_curves.png")
plt.show()


In [39]:

fig, ax = plt.subplots(figsize=(6, 6))
for name, proba in proba_lookup.items():
    curve = utils.get_roc_pr_curve_data(y_test.values, proba)
    ax.plot(curve["recall"], curve["precision"], label=f"{name} (PR-AUC={model_results[name].get('pr_auc', float('nan')):.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves — All Models")
ax.legend()
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "14_pr_curves.png")
plt.show()


In [40]:

fig, axes = plt.subplots(1, len(pred_lookup), figsize=(5 * len(pred_lookup), 4))
for ax, (name, pred) in zip(axes, pred_lookup.items()):
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["BENIGN", "ATTACK"], yticklabels=["BENIGN", "ATTACK"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "14_confusion_matrices.png")
plt.show()


In [41]:

compare_metrics = ["accuracy", "precision", "recall", "f1_score", "false_positive_rate", "roc_auc"]
fig, ax = plt.subplots(figsize=(11, 5))
metrics_df[compare_metrics].plot(kind="bar", ax=ax, colormap="tab10")
ax.set_title("Model Comparison Across Key Metrics (Test Set)")
ax.set_ylabel("Score")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3)
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "14_model_comparison_bars.png")
plt.show()


In [42]:

from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    RandomForestClassifier(n_estimators=200, random_state=CONFIG.random_state, n_jobs=CONFIG.n_jobs),
    X_train_final, y_train_final, cv=3, scoring="f1",
    train_sizes=np.linspace(0.2, 1.0, 5), n_jobs=CONFIG.n_jobs,
)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(train_sizes, train_scores.mean(axis=1), "o-", label="Train F1")
ax.plot(train_sizes, val_scores.mean(axis=1), "o-", label="CV F1")
ax.set_xlabel("Training Set Size")
ax.set_ylabel("F1 Score")
ax.set_title("Random Forest — Learning Curve")
ax.legend()
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "14_learning_curve_rf.png")
plt.show()


In [43]:

from sklearn.model_selection import validation_curve

param_range = [50, 100, 200, 400]
train_scores_vc, val_scores_vc = validation_curve(
    RandomForestClassifier(random_state=CONFIG.random_state, n_jobs=CONFIG.n_jobs),
    X_train_final, y_train_final, param_name="n_estimators", param_range=param_range,
    cv=3, scoring="f1", n_jobs=CONFIG.n_jobs,
)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(param_range, train_scores_vc.mean(axis=1), "o-", label="Train F1")
ax.plot(param_range, val_scores_vc.mean(axis=1), "o-", label="CV F1")
ax.set_xlabel("n_estimators")
ax.set_ylabel("F1 Score")
ax.set_title("Random Forest — Validation Curve")
ax.legend()
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "14_validation_curve_rf.png")
plt.show()



## 15. Explainable AI (SHAP & LIME)

SHAP explains the Random Forest (the strongest / most deployment-realistic tree model)
globally and per-instance; LIME then contrasts a correct vs. an incorrect prediction and a
benign vs. an attack instance, to satisfy the dissertation's analyst-trust objective.


In [44]:

EXPLAIN_MODEL_NAME = "Random Forest"
explain_model = trained_models[EXPLAIN_MODEL_NAME]
# TreeExplainer needs the raw fitted estimator, not the surrounding sklearn Pipeline
explain_model_raw = explain_model.named_steps["classifier"] if hasattr(explain_model, "named_steps") else explain_model

shap_bg = X_train_final.sample(n=min(200, len(X_train_final)), random_state=CONFIG.random_state)
shap_sample = X_test_s.sample(n=min(300, len(X_test_s)), random_state=CONFIG.random_state)

explainer = shap.TreeExplainer(explain_model_raw)
shap_values = explainer.shap_values(shap_sample)
# sklearn>=1.4 / shap>=0.44 returns a (n, features, classes) array for binary RF; take class 1
shap_values_attack = shap_values[..., 1] if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3 else shap_values[1]


In [45]:

plt.figure()
shap.summary_plot(shap_values_attack, shap_sample, show=False)
plt.title(f"SHAP Summary — {EXPLAIN_MODEL_NAME} (Attack class)")
plt.tight_layout()
plt.savefig(CONFIG.figures_dir / "15_shap_summary.png", bbox_inches="tight")
plt.show()


In [46]:

plt.figure()
shap.summary_plot(shap_values_attack, shap_sample, plot_type="bar", show=False)
plt.title(f"SHAP Global Feature Importance (mean |SHAP|) — {EXPLAIN_MODEL_NAME}")
plt.tight_layout()
plt.savefig(CONFIG.figures_dir / "15_shap_bar.png", bbox_inches="tight")
plt.show()


In [47]:

expected_value = explainer.expected_value
expected_value = expected_value[1] if isinstance(expected_value, (list, np.ndarray)) else expected_value

plt.figure()
shap.plots._waterfall.waterfall_legacy(expected_value, shap_values_attack[0], feature_names=shap_sample.columns.tolist(), show=False)
plt.title("SHAP Waterfall — Single Test Instance")
plt.tight_layout()
plt.savefig(CONFIG.figures_dir / "15_shap_waterfall.png", bbox_inches="tight")
plt.show()


In [48]:

top2_features = importances.head(2).index.tolist()
shap.dependence_plot(top2_features[0], shap_values_attack, shap_sample,
                      interaction_index=top2_features[1], show=False)
plt.title(f"SHAP Dependence — {top2_features[0]} vs {top2_features[1]}")
plt.tight_layout()
plt.savefig(CONFIG.figures_dir / "15_shap_dependence.png", bbox_inches="tight")
plt.show()


In [49]:

force_html = shap.force_plot(expected_value, shap_values_attack[0], shap_sample.iloc[0], matplotlib=False)
shap.save_html(str(CONFIG.figures_dir / "15_shap_force_plot.html"), force_html)
print(f"SHAP force plot (interactive) saved to {CONFIG.figures_dir / '15_shap_force_plot.html'}")


SHAP force plot (interactive) saved to figures/15_shap_force_plot.html


In [50]:

lime_explainer = LimeTabularExplainer(
    training_data=X_train_final.values,
    feature_names=X_train_final.columns.tolist(),
    class_names=["BENIGN", "ATTACK"],
    discretize_continuous=True,
    random_state=CONFIG.random_state,
)

test_reset = X_test_s.reset_index(drop=True)
y_test_reset = pd.Series(y_test).reset_index(drop=True)

correct_idx = np.where((rf_pred == y_test.values) & (y_test.values == 1))[0]
incorrect_idx = np.where(rf_pred != y_test.values)[0]
benign_idx = np.where(y_test.values == 0)[0]
attack_idx = np.where(y_test.values == 1)[0]

lime_cases = {
    "correct_attack_prediction": correct_idx[0] if len(correct_idx) else attack_idx[0],
    "incorrect_prediction": incorrect_idx[0] if len(incorrect_idx) else attack_idx[0],
    "benign_traffic": benign_idx[0],
    "attack_traffic": attack_idx[0],
}

for case_name, idx in lime_cases.items():
    exp = lime_explainer.explain_instance(
        test_reset.iloc[idx].values, explain_model.predict_proba, num_features=8,
    )
    fig = exp.as_pyplot_figure()
    fig.suptitle(f"LIME — {case_name.replace('_', ' ').title()}")
    fig.tight_layout()
    fig.savefig(CONFIG.figures_dir / f"15_lime_{case_name}.png", bbox_inches="tight")
    plt.show()
    print(f"{case_name}: true={y_test_reset.iloc[idx]}, predicted={rf_pred[idx]}")


correct_attack_prediction: true=1, predicted=1


incorrect_prediction: true=0, predicted=1


benign_traffic: true=0, predicted=0


attack_traffic: true=1, predicted=1



## 16. Error Analysis

Which attack families the best model misses, and why (using SHAP feature attributions
already computed for context).


In [51]:

best_model_name = metrics_df["f1_score"].astype(float).idxmax()
best_pred = pred_lookup[best_model_name]
print(f"Best model by test F1: {best_model_name}")

error_df = X_test.copy()
error_df["true_binary"] = y_test.values
error_df["pred_binary"] = best_pred
error_df["true_attack_label"] = y_raw.loc[X_test.index].values  # original multiclass label
error_df["error_type"] = np.select(
    [
        (error_df.true_binary == 1) & (error_df.pred_binary == 0),
        (error_df.true_binary == 0) & (error_df.pred_binary == 1),
    ],
    ["False Negative", "False Positive"],
    default="Correct",
)

fn_by_attack = error_df[error_df.error_type == "False Negative"]["true_attack_label"].value_counts()
print("False negatives by original attack family (hardest classes to detect):")
print(fn_by_attack)

fig, ax = plt.subplots(figsize=(8, 4))
if len(fn_by_attack):
    fn_by_attack.plot.bar(ax=ax, color="#c0392b")
ax.set_title(f"False Negatives by Attack Family — {best_model_name}")
ax.set_ylabel("Count")
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "16_false_negatives_by_class.png")
plt.show()

error_df.to_csv(CONFIG.results_dir / "16_error_analysis.csv", index=False)


Best model by test F1: Neural Network
False negatives by original attack family (hardest classes to detect):
Series([], Name: count, dtype: int64)



**Interpretation.** Attack families with the fewest training examples (e.g. `Infiltration`,
`Web Attack` variants — the rarest classes even after imbalance handling, since resampling
was applied at the binary level) are typically the hardest for every model to detect,
because there is simply less behavioural signal for the model to learn from. False
positives tend to concentrate on benign flows whose statistical profile (burstiness,
packet-size variance) resembles low-and-slow scanning traffic. Potential improvements:
per-class-aware oversampling (rather than binary SMOTE), sequence/time-window features
(rather than single-flow statistics) to capture multi-flow attack campaigns, and a
human-in-the-loop review queue specifically for the borderline-probability band
(0.4–0.6) identified by the ROC/PR curves in Section 14.



## 17. Final Comparison


In [52]:

ranking_df = metrics_df.copy()
ranking_df["rank_f1"] = ranking_df["f1_score"].rank(ascending=False)
ranking_df["rank_fpr"] = ranking_df["false_positive_rate"].rank(ascending=True)  # lower FPR is better
ranking_df["combined_rank_score"] = ranking_df["rank_f1"] + ranking_df["rank_fpr"]
ranking_df = ranking_df.sort_values("combined_rank_score")
ranking_df.to_csv(CONFIG.metrics_dir / "17_final_ranking.csv")

print("Final model ranking (lower combined_rank_score = better, weighting F1 and FPR equally "
      "per the dissertation's dual focus on detection performance AND false-alarm minimisation):")
ranking_df[["accuracy", "precision", "recall", "f1_score", "false_positive_rate", "roc_auc",
            "rank_f1", "rank_fpr", "combined_rank_score"]]


Final model ranking (lower combined_rank_score = better, weighting F1 and FPR equally per the dissertation's dual focus on detection performance AND false-alarm minimisation):


,accuracy,precision,recall,f1_score,false_positive_rate,roc_auc,rank_f1,rank_fpr,combined_rank_score
Neural Network,1.000000,1.000000,1.0,1.000000,0.000000,1.0,1.0,1.0,2.0
Random Forest,0.998889,0.998047,1.0,0.999022,0.002571,1.0,3.0,3.0,6.0
SVM,0.998889,0.998047,1.0,0.999022,0.002571,1.0,3.0,3.0,6.0
Ensemble,0.998889,0.998047,1.0,0.999022,0.002571,1.0,3.0,3.0,6.0


In [53]:

BEST_OVERALL_MODEL = ranking_df.index[0]
LOWEST_FPR_MODEL = metrics_df["false_positive_rate"].astype(float).idxmin()
print(f"Best overall model (accuracy/FPR balance): {BEST_OVERALL_MODEL}")
print(f"Lowest false-positive-rate model          : {LOWEST_FPR_MODEL}")


Best overall model (accuracy/FPR balance): Neural Network
Lowest false-positive-rate model          : Neural Network



## 18. Conclusions

*(This cell's printed variables are computed live from your dataset run above — re-read
them after running on the real CICIDS2017 data before writing the dissertation's
discussion chapter; the synthetic smoke-test numbers below are illustrative only.)*

- **Best-performing model** — see `BEST_OVERALL_MODEL` printed in Section 17, ranked
  jointly on F1-score and false positive rate as the dissertation's evaluation criteria
  require.
- **Lowest false positives** — see `LOWEST_FPR_MODEL`; in a banking SOC this is often the
  more operationally important criterion than raw accuracy, since it directly controls
  analyst alert load.
- **Best explainability** — the Random Forest is the most naturally explainable of the
  three base models (TreeExplainer SHAP values are exact and fast), while the Neural
  Network requires the more approximate LIME/KernelSHAP treatment; this is itself a
  deployment consideration for a bank that requires auditable, explainable decisions.
- **Limitations** — (1) CICIDS2017, while widely used, is a 2017 lab-generated capture and
  its attack traffic will not perfectly reflect present-day banking-specific attack
  patterns; (2) this notebook's SVM training set is deliberately subsampled for
  tractability, which may understate SVM's true ceiling on the full dataset; (3) flow-level
  features cannot capture multi-flow / campaign-level attacker behaviour; (4) results here
  were smoke-tested on synthetic data — **conclusions must be re-drawn from a run against
  the real CICIDS2017 dataset**.
- **Future work** — real-time streaming feature extraction (rather than offline flow
  files), online/incremental learning to adapt to concept drift, and evaluation against
  more recent datasets (e.g. CICIDS2018, CSE-CIC-IDS2018) or bank-specific traffic.
- **Banking deployment suitability** — the framework's explicit false-positive-rate
  tracking, model explainability (SHAP/LIME) and modular, reproducible pipeline make it a
  suitable **structured prototype** for a bank's security operations centre to pilot
  against a live traffic mirror before any production deployment decision — consistent
  with the dissertation's design-science scope (a tested prototype, not a production
  deployment).


In [54]:

final_summary = {
    "best_overall_model": BEST_OVERALL_MODEL,
    "lowest_fpr_model": LOWEST_FPR_MODEL,
    "imbalance_strategy_used": BEST_STRATEGY,
    "ensemble_strategy_used": ENSEMBLE_STRATEGY,
    "n_features_selected": len(selected_features),
    "test_set_size": int(len(X_test)),
    "all_model_metrics": model_results,
}
utils.save_json(final_summary, CONFIG.reports_dir / "18_final_summary.json")
print(json.dumps(final_summary, indent=2, default=str))


{
  "best_overall_model": "Neural Network",
  "lowest_fpr_model": "Neural Network",
  "imbalance_strategy_used": "class_weight_balanced",
  "ensemble_strategy_used": "soft_voting",
  "n_features_selected": 19,
  "test_set_size": 900,
  "all_model_metrics": {
    "Random Forest": {
      "accuracy": 0.9988888888888889,
      "precision": 0.998046875,
      "recall": 1.0,
      "f1_score": 0.9990224828934506,
      "false_positive_rate": 0.002570694087403599,
      "false_negative_rate": 0.0,
      "specificity": 0.9974293059125964,
      "sensitivity": 1.0,
      "balanced_accuracy": 0.9987146529562982,
      "matthews_corrcoef": 0.9977380426742712,
      "cohen_kappa": 0.9977354844553589,
      "roc_auc": 1.0,
      "pr_auc": 1.0
    },
    "SVM": {
      "accuracy": 0.9988888888888889,
      "precision": 0.998046875,
      "recall": 1.0,
      "f1_score": 0.9990224828934506,
      "false_positive_rate": 0.002570694087403599,
      "false_negative_rate": 0.0,
      "specificity": 0.997